In [0]:
# ===================================================
# BLOCK 1 — IMPORTS AND TABLE CONFIGURATION (PYTHON)
# ===================================================

from pyspark.sql import functions as F

CATALOG = "semiconplus_portfolio"
SIMULATION_LANDING = (
    f"{CATALOG}.simulation.simulated_retest_events_landing"
)
BRONZE_RETEST = f"{CATALOG}.bronze.simulated_retest_events"
SILVER_RETEST = f"{CATALOG}.silver.simulated_retest_events"

spark.conf.set("spark.sql.session.timeZone", "UTC")
print("Simulated retest Bronze/Silver configuration loaded.")

In [0]:
# ===================================================
# BLOCK 2 — PUBLISH BRONZE WITH LINEAGE (PYTHON)
# ===================================================

bronze_df = (
    spark.table(SIMULATION_LANDING)
    .withColumn("_source_table", F.lit(SIMULATION_LANDING))
    .withColumn("_source_record_id", F.col("retest_event_id"))
    .withColumn("_ingested_at_utc", F.current_timestamp())
    .withColumn("_pipeline_run_id", F.lit("DAY3_RETEST_V1"))
)

(
    bronze_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(BRONZE_RETEST)
)

print(f"Bronze simulated retest rows: {spark.table(BRONZE_RETEST).count():,}")

In [0]:
# ===================================================
# BLOCK 3 — STANDARDIZE AND PUBLISH SILVER (PYTHON)
# ===================================================

silver_df = (
    spark.table(BRONZE_RETEST)
    .select(
        F.col("retest_event_id").cast("string"),
        F.col("retest_timestamp_utc").cast("timestamp"),
        F.col("source_lot_id").cast("string"),
        F.col("device_id").cast("string"),
        F.col("product_group_id").cast("string"),
        F.col("site_id").cast("string"),
        F.col("equipment_id").cast("string"),
        F.col("defect_code").cast("string"),
        F.col("error_code").cast("string"),
        F.col("retest_input_quantity").cast("long"),
        F.col("retest_good_quantity").cast("long"),
        F.col("retest_fail_quantity").cast("long"),
        F.col("retest_test_time_seconds").cast("double"),
        F.col("simulation_seed").cast("long"),
        F.col("simulation_version").cast("string"),
        F.col("simulated_record_flag").cast("boolean"),
        F.col("record_origin").cast("string"),
        F.col("_source_table"),
        F.col("_source_record_id"),
        F.col("_ingested_at_utc"),
        F.current_timestamp().alias("_silver_processed_at_utc"),
    )
)

invalid_silver_rows = silver_df.filter(
    F.col("retest_event_id").isNull()
    | F.col("retest_timestamp_utc").isNull()
    | F.col("source_lot_id").isNull()
    | F.col("retest_input_quantity").isNull()
    | F.col("retest_good_quantity").isNull()
    | F.col("retest_fail_quantity").isNull()
    | (F.col("retest_input_quantity") < 0)
    | (F.col("retest_good_quantity") < 0)
    | (F.col("retest_good_quantity") > F.col("retest_input_quantity"))
    | (
        F.col("retest_input_quantity")
        != F.col("retest_good_quantity") + F.col("retest_fail_quantity")
    )
    | (~F.col("simulated_record_flag"))
).count()

assert invalid_silver_rows == 0, (
    f"Invalid Silver simulated retest rows: {invalid_silver_rows}"
)

(
    silver_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_RETEST)
)

print(f"Silver simulated retest rows: {spark.table(SILVER_RETEST).count():,}")

In [0]:
# ===================================================
# BLOCK 4 — BRONZE/SILVER RECONCILIATION (PYTHON)
# ===================================================

reconciliation = [
    ("SIMULATION_LANDING", spark.table(SIMULATION_LANDING).count()),
    ("BRONZE", spark.table(BRONZE_RETEST).count()),
    ("SILVER", spark.table(SILVER_RETEST).count()),
]

display(spark.createDataFrame(reconciliation, ["dataset", "row_count"]))
assert len({row_count for _, row_count in reconciliation}) == 1
print("Simulated retest Bronze/Silver reconciliation passed.")